# 08 · Pangolin — a second splice model, run locally

**Pangolin** (Zeng & Li 2022, *Genome Biology* 23:103, PMID 35449021,
[github.com/tkzeng/Pangolin](https://github.com/tkzeng/Pangolin)) predicts splice-site
**usage** — how much a site is actually used — rather than whether a position is a splice
site at all. It reports **one 0-1 score** per variant against the **same** 0.5 / 0.2
cut-points as SpliceAI.

### How independent is it from SpliceAI, really?

Less than "a second model" suggests, and the difference matters if you are tempted to
treat agreement between them as two votes.

**The architecture is shared.** The paper says so directly — *"Pangolin's architecture
resembles that used in SpliceAI"*, with *"the addition of multiple outputs to the final
neural network layer... allowing prediction of splice site usage across different
tissues"*. The installed package bears that out: `pangolin/model.py` uses `L = 32`,
`W = [11×8, 21×4, 41×4]` and dilation `AR = [1×4, 4×4, 10×4, 25×4]` — the same stack of
16 dilated residual blocks, and the same ~10 kb receptive field, as SpliceAI-10k. Two
models that see the same window through the same inductive bias are not independent
observers of it.

**The supervision genuinely differs.** SpliceAI was trained on human annotation — *"whether
a dinucleotide is a splice site or not"*. Pangolin was trained on quantitative usage
measured by RNA-seq across **four species** (human, rhesus macaque, rat, mouse) and
**four tissues** (heart, liver, brain, testis). Different target, different data, much
of it non-human.

So agreement between them is **partial** corroboration: it tells you the signal survives
a change of training target and training species, but not that two independent methods
found it. Where they disagree is often the more informative case. This notebook scores
the panel; the systematic comparison belongs in a benchmark, not here.

> Pangolin is **non-commercial** — cite Zeng & Li 2022 (PMID 35449021).

> ✅ **REAL.** The build cell below **runs the actual model locally**: the weights ship
> inside the pip package, and scoring one gene needs only the ~215 kb CFTR reference
> region, not a whole-genome FASTA. I have not found a precomputed Pangolin release to
> download, and it is not in dbNSFP — so running it is the way to get scores, not a
> fallback.
>
> Whatever scope you run, the output is **real model output** and is labelled
> `source='REAL'`. Scoring fewer variants makes the coverage narrower, not the scores
> less real; `DEMO` in this repo means hand-authored illustrative numbers, which these
> never are.
>
> **Version.** Two different things get recorded, because they answer different
> questions. The **model's release** — Zeng & Li 2022 — is the temporal anchor: a variant
> first reported after the model was trained cannot have informed it, which is what makes
> a report-date hold-out meaningful when this score is later used as evidence (the
> circularity argument in [`05_revel.ipynb`](05_revel.ipynb) section 2). The **run
> provenance** — package version, SHA-256 of the twelve weight files loaded, torch build,
> reference region — is what a rerun has to match. Both go into
> `data/pangolin_cftr.release.json`, surfaced as the `pangolin_release` column.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, for toolkit.py
sys.path.insert(0, str(pathlib.Path.cwd()))          # this directory, for pangolin.py
import toolkit as tk
import pandas as pd, numpy as np
%matplotlib inline

## 1 · Getting real Pangolin scores — you run the model

I have not found a precomputed Pangolin release to download, and it is not carried in
dbNSFP, so the scores here come from running the model. That is lighter than it sounds:
the weights are bundled in the pip package, no whole-genome FASTA is required, and a GPU
only makes it faster.

`tk.load_pangolin()` reads whatever the build cell wrote.

**What gets scored.** The default scope is the CFTR2 variant list, because that is the
set of *CFTR* variants with curated GRCh38 coordinates to hand. Three numbers describe
it, and the build cell prints all three so the coverage is auditable rather than assumed:

| | count | why |
|---|---|---|
| CFTR2 variants in the release | **2,097** | every row of the list built in `benchmark/01_cftr2.ipynb` |
| …carrying GRCh38 coordinates | **1,893** | 204 rows are large rearrangements and similar, described by name only, with no `grch38_pos` to score against |
| …actually scored | **1,892** | one remaining event is larger than the ±50 bp aggregation window can speak to |

Rows that cannot be scored are **kept**, with an empty score and a `skip_reason`. A
coverage figure that is short for a stated reason can be audited; one that is silently
short cannot.

Indels are scored as readily as SNVs, because the model pads its delta track by the
ref/alt length difference. Whether that is *useful* is a separate question — a splice
verdict on a frameshift is usually a correct "no splice impact" that says nothing about
why the variant causes disease.

### The build cell, and what `tools/pangolin_build.py` does for it

The cell below is four steps — **pick the targets, load the models, score, write** — but
the machinery underneath is a vendored scoring routine, a one-hot encoder, twelve model
files to load and hash, and a cached reference slice to validate. Inline that buries the
recipe, so it lives in [`pangolin_build.py`](pangolin_build.py) beside this notebook and is imported.
It ships with the repo, nothing in it runs on import, and `torch` is imported lazily
inside the functions that need it — so the module can be read on a machine that has never
installed it.

| function | what it does | why it is not one line |
|---|---|---|
| `load_models()` | loads the 12 bundled models, returns them **plus a SHA-256 per weight file** | the twelve are 4 tissues × 3 replicates whose predictions get averaged; the hashes go in the release stamp, because a swapped or truncated weight file changes every score without changing any version string |
| `load_region()` | fetches and caches the ~215 kb CFTR slice from Ensembl | scoring one gene should not need a whole-genome FASTA |
| `region_start()` | works out where that slice starts, **and checks it** | it takes the integer pair in the FASTA header whose span matches the sequence length, so a cache written by another tool — or a truncated one — fails loudly instead of offsetting every variant silently |
| `score_variant()` | cuts ±5 kb of context, verifies the reference base, builds the alt, returns `max(gain, \|loss\|)` | the ref check is the guard: a coordinate that disagrees with the reference would otherwise produce a confident wrong number |
| `skip_reason()` | says why a variant *cannot* be scored, or `None` | an explicit reason keeps short coverage auditable |
| `write_release_stamp()` | writes the `.release.json` sidecar | records the model's release year **and** the run provenance — see below |
| `one_hot_encode()`, `compute_score()` | **vendored verbatim** from `pangolin/pangolin.py` | keeps scoring identical to upstream, while avoiding that module's top-level `import pyfastx, vcf` |

**Install.** One extra install, needed by no other notebook here:

```bash
pip install "git+https://github.com/tkzeng/Pangolin.git" pyfaidx gffutils torch
```

Coordinates come from `data/cftr2_cftr.csv` (build it with `benchmark/01_cftr2.ipynb`
first), so the model scores the variant it is actually supposed to rather than a
hand-entered guess.

**Why the stamp is written while the model runs.** The model's *release* (Zeng & Li 2022)
is the temporal anchor — it bounds what could have influenced the model, which is what a
later benchmark needs to avoid grading it on variants it may already have seen. The *run*
provenance — package version, weight hashes, torch build, reference region — is what a
rerun must match. Neither can be reconstructed from a finished CSV, so an extract built
before this stamp existed honestly reports `unknown` rather than a guess assembled from
whatever happens to be installed later.

In [2]:
import pangolin_build as pg   # the model machinery described above

PANGOLIN_CSV = pg.DATA_DIR / "pangolin_cftr.csv"
PANGOLIN_RELEASE_JSON = pg.DATA_DIR / "pangolin_cftr.release.json"
CFTR2_CSV = pg.DATA_DIR / "cftr2_cftr.csv"
SCOPE = "cftr2"          # "cftr2" = the whole CFTR2 list; "curated" = the 5 classic
                         # splice alleles, a fast check that the install works.
                         # Both are REAL model output; only the coverage differs.

if PANGOLIN_CSV.exists():
    print(f"already built -> {PANGOLIN_CSV.name} (delete to rebuild, or edit SCOPE above)")
    if not PANGOLIN_RELEASE_JSON.exists():
        print(f"  ! no {PANGOLIN_RELEASE_JSON.name} beside it -- what produced this extract\n"
              "    cannot be established after the fact, so load_pangolin() reports\n"
              f"    pangolin_release as 'unknown'. Delete {PANGOLIN_CSV.name} and rerun to stamp it.")
elif not CFTR2_CSV.exists():
    raise FileNotFoundError(f"{CFTR2_CSV} missing -- run benchmark/01_cftr2.ipynb first "
                            "(Pangolin needs its authoritative GRCh38 coordinates).")
else:
    # 1. pick the targets, and record why any of them cannot be scored
    cf = pd.read_csv(CFTR2_CSV)
    if SCOPE == "curated":
        cf = cf[cf["cdna_name"].isin(tk.A2_KNOWN_CDNA)]
    cf = cf.copy()
    cf["skip_reason"] = [pg.skip_reason(p, r, a) for p, r, a
                         in zip(cf["grch38_pos"], cf["grch38_ref"], cf["grch38_alt"])]
    print(f"scope={SCOPE}: {len(cf):,} CFTR2 variants, "
          f"{int(cf['skip_reason'].isna().sum()):,} scorable")

    # 2. load the twelve models (hashing each weight file), and the reference slice
    r0, seq, region_header = pg.load_region()
    models, weights = pg.load_models()

    # 3. score every scorable variant, keeping the rest with their reason
    rows, done = [], 0
    for _, v in cf.iterrows():
        # pd.isna, NOT `is None`: iterrows turns the column's None into NaN, and
        # `nan is None` is False -- so every variant would look like it already had a
        # reason and nothing would ever be scored, silently and without an error.
        reason = None if pd.isna(v["skip_reason"]) else v["skip_reason"]
        score = None
        pos = int(v["grch38_pos"]) if pd.notna(v["grch38_pos"]) else None
        r_, a_ = str(v["grch38_ref"]), str(v["grch38_alt"])
        if reason is None:
            try:
                score = pg.score_variant(pos, r_, a_, r0, seq, models)
                done += 1
                if SCOPE == "curated" or done % 200 == 0:
                    print(f"  [{done:5,}] {str(v['cdna_name'])[:24]:24} pangolin={score}")
            except Exception as e:
                reason = str(e).split(" -- ")[0]
        # column order matches the extract this notebook's outputs were produced from,
        # so re-running reproduces the same file rather than a differently-shaped one
        rows.append({"cdna_name": v["cdna_name"], "legacy_name": v["legacy_name"],
                     "chrom": "7", "pos": pos,
                     "ref": r_ if pos else None, "alt": a_ if pos else None,
                     "pangolin_score": score, "cftr2_class": v["cftr2_class"],
                     "source": "REAL", "skip_reason": reason})

    # 4. write the extract and stamp the run
    out = pd.DataFrame(rows)
    out.to_csv(PANGOLIN_CSV, index=False)
    scored = int(out["pangolin_score"].notna().sum())
    print(f"\nPangolin written: {scored:,} scored / {len(out):,} targets "
          f"-> {PANGOLIN_CSV.relative_to(pg.DATA_DIR.parent)}")
    print("release stamp ->", PANGOLIN_RELEASE_JSON.name + ":",
          pg.write_release_stamp(PANGOLIN_RELEASE_JSON, weights, region_header,
                                 SCOPE, scored, len(out)))

scope=cftr2: 2,097 CFTR2 variants, 1,892 scorable


  [  200] c.1721C>A                pangolin=0.1269


  [  400] c.2896del                pangolin=0.1076


  [  600] c.1584+2T>C              pangolin=0.5971


  [  800] c.869+1G>C               pangolin=0.8672


...

  [1,200] c.1181T>G                pangolin=0.0816


  [1,400] c.3140-116C>A            pangolin=0.0048


  [1,600] c.4074A>T                pangolin=0.0015


  [1,800] c.2959_2960dup           pangolin=0.1033



Pangolin written: 1,892 scored / 2,097 targets -> data\pangolin_cftr.csv
release stamp -> pangolin_cftr.release.json: Pangolin, Zeng & Li 2022 (Genome Biol 23:103, PMID 35449021); run with pangolin pkg 1.0.2, 12 bundled weight files, torch 2.13.0+cpu on cpu


## 2 · The splice panel, scored by Pangolin

The same fixed panel of famous CFTR **splice** variants runs through both splice
notebooks, so you can follow one set of variants across the series. The variant list is
`tk.A2_KNOWN_CDNA` and the legacy names are `tk.A2_KNOWN_LEGACY`, both in `toolkit.py`;
the scoring is shown inline below.

CFTR2's coordinates are joined on but not printed — its terms forbid republishing any
portion of its content. The counts are facts *about* the list rather than reproductions
of it, so those are shown.

Comparing these scores against SpliceAI's, or against CFTR2's clinical classes, is a
benchmark rather than a tool walkthrough, and it needs care about what each source can
witness — so it lives in the `predict/` pipeline, not here.

In [3]:
pg_scores = tk.load_pangolin()
HIGH, MOD = tk.THRESHOLDS['pangolin']['high'], tk.THRESHOLDS['pangolin']['moderate']

# coverage funnel: targets -> scorable -> scored, so short coverage is visible
scored = pg_scores[pg_scores['pangolin_score'].notna()].copy()
print(f"{len(scored):,} scored / {len(pg_scores):,} targets | "
      f"source: {pg_scores['source'].unique().tolist()}")
print(f"release: {pg_scores['pangolin_release'].iloc[0]}")
if pg_scores['pangolin_score'].isna().any():
    print("\nnot scored, by reason:")
    print(pg_scores.loc[pg_scores['pangolin_score'].isna(), 'skip_reason']
          .value_counts().to_string())

scored['tier'] = scored['pangolin_score'].apply(
    lambda s: 'HIGH' if s >= HIGH else ('MODERATE' if s >= MOD else 'LOW'))
print("\ntier distribution:")
print(scored['tier'].value_counts().to_string())

# the classic CF splice alleles, recovered from the full run
panel = scored[scored['cdna_name'].isin(tk.A2_KNOWN_CDNA)].copy()
panel['legacy_name'] = panel['cdna_name'].map(tk.A2_KNOWN_LEGACY)
print("\nclassic CF splice alleles:")
panel[['cdna_name', 'legacy_name', 'pangolin_score', 'tier', 'source']].reset_index(drop=True)

1,892 scored / 2,097 targets | source: ['REAL']
release: Pangolin, Zeng & Li 2022 (Genome Biol 23:103, PMID 35449021); run with pangolin pkg 1.0.2, 12 bundled weight files, torch 2.13.0+cpu on cpu

not scored, by reason:
skip_reason
no GRCh38 coordinates in CFTR2    204
event larger than 100 bp            1

tier distribution:
tier
LOW         1518
HIGH         260
MODERATE     114

classic CF splice alleles:


,cdna_name,legacy_name,pangolin_score,tier,source
0,c.3718-2477C>T,3849+10kbC>T,0.3327,MODERATE,REAL
1,c.2657+5G>A,2789+5G>A,0.8194,HIGH,REAL
2,c.3140-26A>G,3272-26A>G,0.8120,HIGH,REAL
3,c.2988+1G>A,3120+1G>A,0.8568,HIGH,REAL
4,c.1680-886A>G,1811+1634A>G,0.7057,HIGH,REAL


## 3 · Key takeaways

1. **Pangolin scores are produced, not downloaded.** I have not found a precomputed
   release, and it is not in dbNSFP — the build cell runs the model locally, which needs
   only the bundled weights and a ~215 kb reference slice.
2. **It is not an independent second opinion on SpliceAI.** The architecture is the same
   stack of 16 dilated residual blocks with the same ~10 kb receptive field; what differs
   is the supervision — quantitative splice-site *usage* across four species and four
   tissues, versus SpliceAI's binary annotated-site target. Agreement is partial
   corroboration, not two votes.
3. **One 0-1 score, same 0.5 / 0.2 cut-points** — but that score is a **collapse**:
   `max(gain, |loss|)` over the ±50 bp window. The direction is discarded before it
   reaches the CSV, so it cannot say whether a site is being created or destroyed. When
   you need the mechanism, read `07_spliceai.ipynb`'s four deltas.
4. **Coverage is stated, not implied.** 2,097 CFTR2 variants → 1,893 with GRCh38
   coordinates → 1,892 scored; the rest are kept with a `skip_reason`.
5. **Every scope is REAL.** Scoring 5 variants instead of 1,892 narrows the coverage, not
   the authenticity — `DEMO` in this repo means hand-authored numbers, which model output
   never is.
6. **Two versions, two purposes.** The model's release year anchors temporal reasoning —
   it bounds what could have influenced the model, which is what a later benchmark needs
   to avoid grading it on what it has already seen. The run provenance (package version,
   weight hashes, torch build, reference region) is what a rerun must match. Both are
   written while the model runs, because neither survives in a finished CSV.
7. Correct citation: **Zeng & Li 2022, PMID 35449021**. Non-commercial.

**This is the last notebook in the published series.** The live CADD API notebook and the
cross-tool benchmark over the whole CFTR2 list are written but held back pending the same
audit pass these notebooks have been through.